# 08 marker annotation

**Purpose:** Assign marker-supported draft cell annotations.

**Original analysis notebook:** `08_marker_based_annotation_HVG8000_r0_6_fullgene.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# 24 — Marker-based annotation draft after HVG8000/scVI r0.6

This notebook performs the first marker-based annotation pass for the 4-dataset GBM core object.

**Main clustering key:** `leiden_scvi_main`  
**Underlying clustering:** `leiden_scvi_HVG8000_r0_6`  
**Input object:** full-gene object with scVI clusters transferred.

Main principles:

1. Use the **full-gene object** for marker interpretation, not the HVG object.
2. Preserve raw counts in `adata.layers['counts']`.
3. Treat tumor-like labels as **putative** until CNV confirms malignancy.
4. Flag small, patient-dominant, sample-dominant, dataset-dominant, and mixed/ambiguous clusters.
5. Save a new h5ad for downstream notebooks and replace outputs if they already exist.


In [ ]:

# =========================
# Imports and paths
# =========================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURES_DIR = PROJECT_DIR / "figures"
RESULTS_DIR = PROJECT_DIR / "results"

for d in [PROCESSED_DIR, METADATA_DIR, FIGURES_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

INTEGRATED_DIR = PROCESSED_DIR / "integrated"

FULLGENE_INPUT_H5AD = INTEGRATED_DIR / "GBM_core_4datasets_fullgene_with_HVG8000_scVI_clusters.h5ad"

MARKER_DE_TOP100_TSV = METADATA_DIR / "marker_DE_leiden_scvi_main_HVG8000_r0_6_wilcoxon_top100_up_down.tsv"
MARKER_DE_TOP20_TSV = METADATA_DIR / "marker_DE_leiden_scvi_main_HVG8000_r0_6_wilcoxon_top20_up.tsv"
CLUSTER_QC_TSV = METADATA_DIR / "marker_DE_HVG8000_r0_6_cluster_QC_summary.tsv"
CLUSTER_COMPOSITION_TSV = METADATA_DIR / "marker_DE_HVG8000_r0_6_cluster_composition_summary.tsv"
MARKER_PANEL_PRESENCE_TSV = METADATA_DIR / "marker_DE_HVG8000_r0_6_marker_panel_presence.tsv"

ANNOTATION_TSV = METADATA_DIR / "annotation_marker_based_HVG8000_r0_6.tsv"
ANNOTATION_EVIDENCE_TSV = METADATA_DIR / "cluster_marker_annotation_evidence_HVG8000_r0_6.tsv"
ANNOTATION_COUNTS_TSV = METADATA_DIR / "annotation_marker_based_HVG8000_r0_6_cell_counts.tsv"
ANNOTATION_COMPOSITION_TSV = METADATA_DIR / "annotation_marker_based_HVG8000_r0_6_composition_annotated.tsv"
ANNOTATION_BIAS_TSV = METADATA_DIR / "annotation_marker_based_HVG8000_r0_6_bias_flags.tsv"
ANNOTATION_DECISION_REPORT_TSV = METADATA_DIR / "annotation_marker_based_HVG8000_r0_6_decision_report.tsv"
ANNOTATION_OUTPUT_MANIFEST_TSV = METADATA_DIR / "annotation_marker_based_HVG8000_r0_6_output_manifest.tsv"

ANNOTATED_H5AD = INTEGRATED_DIR / "GBM_core_4datasets_fullgene_HVG8000_r0_6_marker_based_annotation.h5ad"
ANNOTATED_H5AD_ALIAS = INTEGRATED_DIR / "GBM_core_4datasets_fullgene_marker_based_annotation.h5ad"

FIG_DIR = FIGURES_DIR / "annotation_marker_based_HVG8000_r0_6"
FIG_DIR.mkdir(parents=True, exist_ok=True)

CLUSTER_KEY = "leiden_scvi_main"
SOURCE_CLUSTER_KEY = "leiden_scvi_HVG8000_r0_6"

ANNOTATION_COL = "annotation_marker_based_HVG8000_r0_6"
LINEAGE_COL = "lineage_marker_based_HVG8000_r0_6"
CONFIDENCE_COL = "annotation_confidence_HVG8000_r0_6"
MALIGNANCY_COL = "malignancy_status_marker_based_HVG8000_r0_6"
CAUTION_COL = "annotation_caution_HVG8000_r0_6"
NOTE_COL = "annotation_note_HVG8000_r0_6"

sc.settings.figdir = str(FIG_DIR)
sc.settings.verbosity = 2

print("Project:", PROJECT_DIR)
print("Input h5ad:", FULLGENE_INPUT_H5AD)
print("Cluster key:", CLUSTER_KEY)


In [ ]:

# =========================
# Helper functions
# =========================

def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} shape={df.shape}")


def write_h5ad_replace(adata_obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    adata_obj.write_h5ad(path)
    print(f"[SAVED] {path}")


def clean_old_rank_keys(adata_obj):
    for k in list(adata_obj.uns.keys()):
        if str(k).startswith("rank_genes_groups"):
            del adata_obj.uns[k]
            print(f"[REMOVED old uns key] {k}")


def safe_read_tsv(path, required=True):
    path = Path(path)
    if not path.exists():
        if required:
            raise FileNotFoundError(path)
        print(f"[WARNING] missing optional file: {path}")
        return pd.DataFrame()
    df = pd.read_csv(path, sep="\t")
    print(f"[READ] {path} shape={df.shape}")
    return df


def list_to_string(x):
    if isinstance(x, (list, tuple, set)):
        return ";".join(map(str, x))
    return str(x)


def present_genes(genes, var_names):
    var_set = set(map(str, var_names))
    return [g for g in genes if g in var_set]


def get_cluster_order(labels):
    vals = pd.Series(labels).astype(str).unique().tolist()
    def keyfun(x):
        return (0, int(x)) if str(x).isdigit() else (1, str(x))
    return sorted(vals, key=keyfun)


In [ ]:

# =========================
# Read inputs
# =========================

if not FULLGENE_INPUT_H5AD.exists():
    raise FileNotFoundError(FULLGENE_INPUT_H5AD)

adata = sc.read_h5ad(FULLGENE_INPUT_H5AD)
print(adata)
print("layers:", list(adata.layers.keys()))
print("obsm:", list(adata.obsm.keys()))
print("obs columns preview:", list(adata.obs.columns)[:80])

if CLUSTER_KEY not in adata.obs.columns:
    if SOURCE_CLUSTER_KEY in adata.obs.columns:
        adata.obs[CLUSTER_KEY] = adata.obs[SOURCE_CLUSTER_KEY].astype(str)
        print(f"[ADDED] {CLUSTER_KEY} from {SOURCE_CLUSTER_KEY}")
    else:
        raise KeyError(f"Neither {CLUSTER_KEY} nor {SOURCE_CLUSTER_KEY} exists in adata.obs")

adata.obs[CLUSTER_KEY] = adata.obs[CLUSTER_KEY].astype(str).astype("category")
cluster_order = get_cluster_order(adata.obs[CLUSTER_KEY])
print("n clusters:", len(cluster_order))
print("clusters:", cluster_order)

if "counts" not in adata.layers:
    print("[WARNING] counts layer is absent in full-gene object. This is not ideal for downstream full-gene analyses.")
else:
    print("counts layer is present:", adata.layers["counts"].shape)

marker_de = safe_read_tsv(MARKER_DE_TOP100_TSV, required=True)
marker_top20 = safe_read_tsv(MARKER_DE_TOP20_TSV, required=False)
cluster_qc = safe_read_tsv(CLUSTER_QC_TSV, required=False)
cluster_comp = safe_read_tsv(CLUSTER_COMPOSITION_TSV, required=False)
marker_panel_presence = safe_read_tsv(MARKER_PANEL_PRESENCE_TSV, required=False)

marker_de["cluster"] = marker_de["cluster"].astype(str)
if not marker_top20.empty:
    marker_top20["cluster"] = marker_top20["cluster"].astype(str)
if not cluster_qc.empty:
    cluster_qc["cluster"] = cluster_qc["cluster"].astype(str)
if not cluster_comp.empty:
    cluster_comp["cluster"] = cluster_comp["cluster"].astype(str)


## Marker panels used for evidence scoring

These marker panels are not used as a black-box classifier. They are used to summarize evidence and make the manual annotation table auditable.


In [ ]:

# =========================
# Marker panels
# =========================

MARKER_PANELS = {
    "Tumor_AC_like": ["GFAP", "AQP4", "ALDH1L1", "SLC1A3", "APOE", "CLU", "SPARCL1", "SLC1A2", "FABP7"],
    "Tumor_OPC_like": ["PDGFRA", "OLIG1", "OLIG2", "SOX10", "CSPG4", "BCAN", "PTPRZ1", "PTN", "DLL3"],
    "Tumor_NPC_like": ["SOX4", "SOX11", "NES", "ASCL1", "DCX", "STMN2", "TUBB3", "MYT1L", "DLX5"],
    "Tumor_MES_like": ["CD44", "CHI3L1", "SERPINE1", "VIM", "FN1", "TGFBI", "ANXA1", "LGALS3", "CTHRC1"],
    "Hypoxia_stress": ["CA9", "VEGFA", "HILPDA", "LOX", "BNIP3", "NDRG1", "ADM", "DDIT4"],
    "Cycling": ["MKI67", "TOP2A", "UBE2C", "BIRC5", "PBK", "CENPF", "NUSAP1", "CDK1"],
    "Myeloid_TAM": ["LYZ", "TYROBP", "AIF1", "C1QA", "C1QB", "C1QC", "TREM2", "CSF1R", "APOE", "CD68", "CD163"],
    "Inflammatory_myeloid": ["IL1B", "CCL3", "CCL4", "S100A8", "S100A9", "FCN1", "AQP9", "FPR2", "LILRA5"],
    "T_NK": ["CD3D", "CD3E", "TRAC", "LCK", "NKG7", "GNLY", "GZMB", "PRF1", "KLRD1"],
    "B_Plasma": ["CD79A", "CD79B", "MS4A1", "MZB1", "JCHAIN", "IGHM", "IGKC", "POU2AF1", "SDC1"],
    "Oligodendrocyte": ["MOG", "MAG", "MOBP", "PLP1", "MBP", "ERMN", "CLDN11"],
    "Endothelial": ["PECAM1", "VWF", "KDR", "FLT1", "ESAM", "EMCN", "CLEC14A", "ERG"],
    "Mural_Pericyte": ["PDGFRB", "ACTA2", "RGS5", "MCAM", "CSPG4", "NOTCH3", "ENPEP", "ABCC9"],
    "Fibroblast_ECM": ["COL1A1", "COL1A2", "COL3A1", "LUM", "DCN", "CTHRC1", "COL6A1", "COL6A2"],
    "Erythroid_Hb": ["HBA1", "HBA2", "HBB", "ALAS2"],
    "Mast": ["KIT", "TPSAB1", "TPSB2", "CPA3", "MS4A2", "GATA2", "RGS13", "ENPP3"],
}

panel_rows = []
for panel, genes in MARKER_PANELS.items():
    present = present_genes(genes, adata.var_names)
    missing = [g for g in genes if g not in present]
    panel_rows.append({
        "panel": panel,
        "n_panel_genes": len(genes),
        "n_present": len(present),
        "n_missing": len(missing),
        "present_genes": ";".join(present),
        "missing_genes": ";".join(missing),
    })

panel_presence_df = pd.DataFrame(panel_rows)
PANEL_PRESENCE_TSV = METADATA_DIR / "annotation_marker_panel_presence_HVG8000_r0_6.tsv"
write_tsv_replace(panel_presence_df, PANEL_PRESENCE_TSV)
display(panel_presence_df)


In [ ]:

# =========================
# Summarize DE marker evidence per cluster
# =========================

required_cols = {"cluster", "gene", "direction", "logfoldchange", "padj", "pct_in_cluster", "pct_rest"}
missing_cols = required_cols - set(marker_de.columns)
if missing_cols:
    raise ValueError(f"marker_de is missing required columns: {missing_cols}")

up_de = marker_de[marker_de["direction"].astype(str).eq("up")].copy()
down_de = marker_de[marker_de["direction"].astype(str).eq("down")].copy()

# Make sure top genes are ordered by direction rank where available.
rank_col = "direction_rank" if "direction_rank" in marker_de.columns else "rank_scanpy"
up_de = up_de.sort_values(["cluster", rank_col])
down_de = down_de.sort_values(["cluster", rank_col])

def top_genes_for_cluster(df, cl, n=20):
    sub = df[df["cluster"].astype(str) == str(cl)].sort_values(rank_col)
    return sub["gene"].astype(str).head(n).tolist()

# panel hit scoring from top100 up genes
rows = []
for cl in cluster_order:
    top20_up = top_genes_for_cluster(up_de, cl, n=20)
    top100_up = top_genes_for_cluster(up_de, cl, n=100)
    top20_down = top_genes_for_cluster(down_de, cl, n=20)
    top100_set = set(top100_up)

    panel_hits = {}
    for panel, genes in MARKER_PANELS.items():
        hits = [g for g in genes if g in top100_set]
        panel_hits[panel] = hits

    best_panels = sorted(panel_hits.items(), key=lambda kv: len(kv[1]), reverse=True)
    best_panel_summary = ";".join([f"{p}:{len(h)}" for p, h in best_panels if len(h) > 0])

    row = {
        "cluster": str(cl),
        "top20_up_genes": ";".join(top20_up),
        "top20_down_genes": ";".join(top20_down),
        "panel_hit_summary_top100_up": best_panel_summary,
    }
    for panel, hits in panel_hits.items():
        row[f"{panel}_n_hits_top100_up"] = len(hits)
        row[f"{panel}_hits_top100_up"] = ";".join(hits)
    rows.append(row)

evidence_auto = pd.DataFrame(rows)

display(evidence_auto[["cluster", "top20_up_genes", "panel_hit_summary_top100_up"]])


## Manual marker-based annotation draft

This table is intentionally conservative. Tumor-like calls remain **putative** until CNV validation. Rare or mixed clusters are flagged rather than forced into a confident label.


In [ ]:

# =========================
# Manual annotation map
# =========================

annotation_records = [
    {
        "cluster": "0",
        "annotation": "putative_tumor_OPC_NPC_like",
        "lineage": "putative_tumor_like",
        "malignancy_status": "putative_tumor_like_requires_CNV",
        "confidence": "moderate_high",
        "note": "OPC/NPC-like markers including PTPRZ1, BCAN, OLIG1, PTN. Malignancy requires CNV confirmation.",
    },
    {
        "cluster": "1",
        "annotation": "myeloid_C1Q_TREM2_APOE_TAM",
        "lineage": "myeloid",
        "malignancy_status": "non_malignant_reference_candidate",
        "confidence": "high",
        "note": "C1QC/C1QB/TREM2/CSF1R/APOE-like TAM/macrophage program.",
    },
    {
        "cluster": "2",
        "annotation": "cycling_proliferating_cells",
        "lineage": "cycling_ambiguous",
        "malignancy_status": "requires_marker_context_and_CNV",
        "confidence": "moderate",
        "note": "Strong cell-cycle markers TOP2A/PBK/BIRC5/UBE2C. Underlying lineage requires marker context and CNV.",
    },
    {
        "cluster": "3",
        "annotation": "myeloid_inflammatory_MES_associated",
        "lineage": "myeloid",
        "malignancy_status": "non_malignant_or_MES_associated_requires_CNV_check",
        "confidence": "moderate",
        "note": "TGFBI/S100A4/TYROBP/LYZ suggest inflammatory myeloid/MES-associated macrophage-like cluster; monitor MES ambiguity.",
    },
    {
        "cluster": "4",
        "annotation": "putative_tumor_hypoxia_stress",
        "lineage": "putative_tumor_like_stress",
        "malignancy_status": "putative_tumor_like_requires_CNV",
        "confidence": "moderate_high",
        "note": "CA9/VEGFA/HILPDA/LOX hypoxia stress program; likely tumor-associated but CNV required.",
    },
    {
        "cluster": "5",
        "annotation": "T_NK_cells",
        "lineage": "lymphoid",
        "malignancy_status": "non_malignant_immune",
        "confidence": "high",
        "note": "CD3D/CD3E/TRAC/GZMA/LCK T/NK markers. Check rGBM/DS3 dominance as biological or sampling bias.",
    },
    {
        "cluster": "6",
        "annotation": "myeloid_APC_HLA_CD163_TAM",
        "lineage": "myeloid",
        "malignancy_status": "non_malignant_reference_candidate",
        "confidence": "high",
        "note": "HLA-DRA/HLA-DPB1/CD74/CD163 antigen-presenting macrophage/TAM program.",
    },
    {
        "cluster": "7",
        "annotation": "putative_tumor_AC_like_reactive",
        "lineage": "putative_tumor_like",
        "malignancy_status": "putative_tumor_like_requires_CNV",
        "confidence": "moderate_high",
        "note": "AQP4/GFAP/CLU/SPARCL1 astrocytic/reactive AC-like program; tumor vs normal/reactive requires CNV.",
    },
    {
        "cluster": "8",
        "annotation": "oligodendrocyte_myelinating",
        "lineage": "normal_brain_glial",
        "malignancy_status": "non_malignant_reference_candidate",
        "confidence": "high",
        "note": "MAG/MOG/MOBP/ERMN myelinating oligodendrocyte markers.",
    },
    {
        "cluster": "9",
        "annotation": "ambiguous_mixed_Hb_T_NK_HLA",
        "lineage": "ambiguous_mixed_or_doublet",
        "malignancy_status": "exclude_or_review_before_final_calls",
        "confidence": "low",
        "note": "Mixed HBB/HBA, GNLY, and HLA/myeloid signals; possible mixed/doublet/ambient or condition-specific artifact. Requires manual review.",
    },
    {
        "cluster": "10",
        "annotation": "myeloid_FCN1_inflammatory_monocyte_like",
        "lineage": "myeloid",
        "malignancy_status": "non_malignant_immune",
        "confidence": "moderate_high",
        "note": "FCN1/AQP9/FPR2/LILRA5 inflammatory monocyte-like markers; assess rGBM/DS3 dominance.",
    },
    {
        "cluster": "11",
        "annotation": "putative_tumor_NPC_neuronal_like",
        "lineage": "putative_tumor_like",
        "malignancy_status": "putative_tumor_like_requires_CNV",
        "confidence": "moderate",
        "note": "STMN2/DLX5/DCX/MYT1L neural/NPC-like markers. CNV needed to distinguish tumor-like from neuronal-lineage contamination.",
    },
    {
        "cluster": "12",
        "annotation": "MES_ECM_stromal_like_ambiguous",
        "lineage": "mesenchymal_stromal_ambiguous",
        "malignancy_status": "requires_CNV_and_stromal_marker_review",
        "confidence": "moderate_low",
        "note": "COL1A1/COL3A1/CTHRC1/FABP7 ECM/MES-like signal. Could be stromal/perivascular or MES tumor-like; CNV required.",
    },
    {
        "cluster": "13",
        "annotation": "mural_pericyte_fibroblast_like",
        "lineage": "vascular_stromal",
        "malignancy_status": "non_malignant_stromal_candidate",
        "confidence": "moderate_high",
        "note": "PDGFRB/CD248/LUM/ENPEP pericyte/mural/fibroblast-like markers.",
    },
    {
        "cluster": "14",
        "annotation": "endothelial",
        "lineage": "vascular_endothelial",
        "malignancy_status": "non_malignant_vascular",
        "confidence": "high",
        "note": "KDR/ERG/ESAM/EMCN/CLEC14A endothelial markers.",
    },
    {
        "cluster": "15",
        "annotation": "B_plasma_like",
        "lineage": "lymphoid_B_plasma",
        "malignancy_status": "non_malignant_immune",
        "confidence": "moderate_high",
        "note": "CD79A/POU2AF1/MZB1/IGHM B/plasma-like markers; MS4A1/JCHAIN may be absent from object.",
    },
    {
        "cluster": "16",
        "annotation": "rare_mast_cell_like",
        "lineage": "rare_immune_mast_like",
        "malignancy_status": "non_malignant_immune_review",
        "confidence": "moderate_low",
        "note": "KIT/GATA2/RGS13/ENPP3 mast-like markers; very small cluster, review carefully.",
    },
    {
        "cluster": "17",
        "annotation": "erythroid_Hb_high_or_ambient_RBC",
        "lineage": "erythroid_or_ambient",
        "malignancy_status": "exclude_or_review_before_final_calls",
        "confidence": "moderate",
        "note": "HBA1/HBA2/HBB hemoglobin-high cluster; likely RBC/ambient contamination or rare erythroid cells. Very small cluster.",
    },
]

annotation_table = pd.DataFrame(annotation_records)
annotation_table["cluster"] = annotation_table["cluster"].astype(str)

# Ensure every observed cluster has an annotation row.
missing_ann = sorted(set(cluster_order) - set(annotation_table["cluster"]), key=lambda x: int(x) if str(x).isdigit() else str(x))
if missing_ann:
    raise ValueError(f"Missing annotation records for clusters: {missing_ann}")

extra_ann = sorted(set(annotation_table["cluster"]) - set(cluster_order))
if extra_ann:
    print("[WARNING] annotation table contains clusters not observed:", extra_ann)

annotation_table = annotation_table.sort_values("cluster", key=lambda s: s.astype(str).map(lambda x: int(x) if x.isdigit() else 999999))
display(annotation_table)


In [ ]:

# =========================
# Compute bias / composition flags from cluster QC
# =========================

if cluster_qc.empty:
    print("[INFO] cluster_qc file missing; computing basic QC from adata.obs")
    rows = []
    for cl in cluster_order:
        obs_sub = adata.obs[adata.obs[CLUSTER_KEY].astype(str) == str(cl)]
        row = {"cluster": str(cl), "n_cells": int(obs_sub.shape[0])}
        for col in ["core_dataset", "condition", "patient_id", "sample_id"]:
            if col in obs_sub.columns:
                vc = obs_sub[col].astype(str).value_counts(dropna=False)
                row[f"top_{col}"] = vc.index[0] if len(vc) else "NA"
                row[f"top_{col}_fraction"] = float(vc.iloc[0] / obs_sub.shape[0]) if len(vc) and obs_sub.shape[0] > 0 else np.nan
                row[f"n_{col}"] = int(vc.shape[0])
        rows.append(row)
    cluster_qc_use = pd.DataFrame(rows)
else:
    cluster_qc_use = cluster_qc.copy()

# Normalize column names from existing QC table.
rename_map = {
    "top_core_dataset": "top_core_dataset",
    "top_core_dataset_fraction": "top_core_dataset_fraction",
    "top_condition": "top_condition",
    "top_condition_fraction": "top_condition_fraction",
    "top_patient_id": "top_patient_id",
    "top_patient_id_fraction": "top_patient_id_fraction",
    "top_sample_id": "top_sample_id",
    "top_sample_id_fraction": "top_sample_id_fraction",
}
cluster_qc_use = cluster_qc_use.rename(columns=rename_map)
cluster_qc_use["cluster"] = cluster_qc_use["cluster"].astype(str)

# Create standard flags.
def bool_col(df, col, default=False):
    return df[col].astype(bool) if col in df.columns else pd.Series(default, index=df.index)

bias = cluster_qc_use.copy()
if "n_cells" not in bias.columns:
    bias["n_cells"] = bias["cluster"].map(adata.obs[CLUSTER_KEY].astype(str).value_counts().to_dict()).astype(int)

bias["flag_small_lt_500"] = bias["n_cells"].astype(int) < 500
bias["flag_small_lt_100"] = bias["n_cells"].astype(int) < 100
bias["flag_dataset_dominant_gt_0_80"] = bias.get("top_core_dataset_fraction", pd.Series(0, index=bias.index)).astype(float) > 0.80
bias["flag_condition_dominant_gt_0_80"] = bias.get("top_condition_fraction", pd.Series(0, index=bias.index)).astype(float) > 0.80
bias["flag_patient_dominant_gt_0_80"] = bias.get("top_patient_id_fraction", pd.Series(0, index=bias.index)).astype(float) > 0.80
bias["flag_sample_dominant_gt_0_80"] = bias.get("top_sample_id_fraction", pd.Series(0, index=bias.index)).astype(float) > 0.80

flag_cols = [c for c in bias.columns if c.startswith("flag_")]
bias["n_bias_or_size_flags"] = bias[flag_cols].sum(axis=1).astype(int)
bias["bias_flag_summary"] = bias[flag_cols].apply(lambda r: ";".join([c for c, v in r.items() if bool(v)]) if any(r) else "none", axis=1)

write_tsv_replace(bias, ANNOTATION_BIAS_TSV)
display(bias[["cluster", "n_cells", "bias_flag_summary"] + [c for c in ["top_core_dataset", "top_core_dataset_fraction", "top_condition", "top_condition_fraction", "top_patient_id", "top_patient_id_fraction", "top_sample_id", "top_sample_id_fraction"] if c in bias.columns]])


In [ ]:

# =========================
# Build final annotation evidence table
# =========================

ann = annotation_table.merge(evidence_auto, on="cluster", how="left")
ann = ann.merge(bias[["cluster", "n_cells", "bias_flag_summary", "n_bias_or_size_flags"] + [c for c in bias.columns if c.startswith("top_")]], on="cluster", how="left")

# Caution rules combine manual notes and QC flags.
def assign_caution(row):
    caution = []
    if str(row.get("confidence", "")).lower() in {"low", "moderate_low"}:
        caution.append("low_or_moderate_low_confidence")
    if str(row.get("malignancy_status", "")).startswith("putative_tumor") or "requires_CNV" in str(row.get("malignancy_status", "")):
        caution.append("requires_CNV")
    if row.get("bias_flag_summary", "none") != "none":
        caution.append(str(row.get("bias_flag_summary")))
    if "ambiguous" in str(row.get("lineage", "")) or "ambiguous" in str(row.get("annotation", "")):
        caution.append("ambiguous_annotation")
    if "exclude_or_review" in str(row.get("malignancy_status", "")):
        caution.append("review_or_exclude")
    return ";".join(dict.fromkeys([c for c in caution if c])) if caution else "none"

ann["annotation_caution"] = ann.apply(assign_caution, axis=1)

write_tsv_replace(annotation_table, ANNOTATION_TSV)
write_tsv_replace(ann, ANNOTATION_EVIDENCE_TSV)

display(ann[["cluster", "annotation", "lineage", "malignancy_status", "confidence", "n_cells", "bias_flag_summary", "annotation_caution", "top20_up_genes"]])


In [ ]:

# =========================
# Apply annotation to adata.obs and save counts tables
# =========================

annotation_map = dict(zip(annotation_table["cluster"], annotation_table["annotation"]))
lineage_map = dict(zip(annotation_table["cluster"], annotation_table["lineage"]))
confidence_map = dict(zip(annotation_table["cluster"], annotation_table["confidence"]))
malignancy_map = dict(zip(annotation_table["cluster"], annotation_table["malignancy_status"]))
note_map = dict(zip(annotation_table["cluster"], annotation_table["note"]))
caution_map = dict(zip(ann["cluster"], ann["annotation_caution"]))

cl = adata.obs[CLUSTER_KEY].astype(str)
adata.obs[ANNOTATION_COL] = cl.map(annotation_map).fillna("unannotated").astype("category")
adata.obs[LINEAGE_COL] = cl.map(lineage_map).fillna("unannotated").astype("category")
adatas_conf = cl.map(confidence_map).fillna("unannotated")
adata.obs[CONFIDENCE_COL] = adatas_conf.astype("category")
adata.obs[MALIGNANCY_COL] = cl.map(malignancy_map).fillna("unannotated").astype("category")
adata.obs[CAUTION_COL] = cl.map(caution_map).fillna("none").astype("category")
adata.obs[NOTE_COL] = cl.map(note_map).fillna("").astype(str)

counts = (
    adata.obs
    .groupby([CLUSTER_KEY, ANNOTATION_COL, LINEAGE_COL, CONFIDENCE_COL, MALIGNANCY_COL, CAUTION_COL], observed=True)
    .size()
    .reset_index(name="n_cells")
    .sort_values(CLUSTER_KEY, key=lambda s: s.astype(str).map(lambda x: int(x) if x.isdigit() else 999999))
)
counts["fraction_total"] = counts["n_cells"] / adata.n_obs
write_tsv_replace(counts, ANNOTATION_COUNTS_TSV)
display(counts)

# Annotated composition table
if not cluster_comp.empty:
    comp_annot = cluster_comp.merge(annotation_table, on="cluster", how="left")
    write_tsv_replace(comp_annot, ANNOTATION_COMPOSITION_TSV)
    display(comp_annot.head(20))
else:
    print("[INFO] no cluster composition table found; skipping annotated composition table")


In [ ]:

# =========================
# Decision report
# =========================

decision_rows = [
    {"item": "input_h5ad", "value": str(FULLGENE_INPUT_H5AD)},
    {"item": "output_h5ad", "value": str(ANNOTATED_H5AD)},
    {"item": "output_h5ad_alias", "value": str(ANNOTATED_H5AD_ALIAS)},
    {"item": "cluster_key_main", "value": CLUSTER_KEY},
    {"item": "source_cluster_key", "value": SOURCE_CLUSTER_KEY},
    {"item": "n_cells", "value": str(adata.n_obs)},
    {"item": "n_genes", "value": str(adata.n_vars)},
    {"item": "n_clusters", "value": str(len(cluster_order))},
    {"item": "annotation_col", "value": ANNOTATION_COL},
    {"item": "lineage_col", "value": LINEAGE_COL},
    {"item": "confidence_col", "value": CONFIDENCE_COL},
    {"item": "malignancy_col", "value": MALIGNANCY_COL},
    {"item": "caution_col", "value": CAUTION_COL},
    {"item": "principle", "value": "Tumor-like labels are putative until CNV validation."},
    {"item": "main_recommendation", "value": "Proceed to marker review / annotation refinement, then CNV-based malignant validation."},
]

decision_report = pd.DataFrame(decision_rows)
write_tsv_replace(decision_report, ANNOTATION_DECISION_REPORT_TSV)
display(decision_report)


## Visual checks

The plots below are optional but useful. They should not be the only basis for annotation. Use them together with DE, marker panels, and composition/bias flags.


In [ ]:

# =========================
# UMAP plots for annotation / lineage / confidence / malignancy / caution
# =========================

if "X_umap" not in adata.obsm:
    print("[WARNING] X_umap not found; skipping UMAP plots")
else:
    for color_col in [ANNOTATION_COL, LINEAGE_COL, CONFIDENCE_COL, MALIGNANCY_COL, CAUTION_COL, CLUSTER_KEY, "core_dataset", "condition"]:
        if color_col in adata.obs.columns:
            sc.pl.umap(
                adata,
                color=color_col,
                legend_loc="right margin",
                frameon=False,
                size=3,
                show=False,
            )
            fig = plt.gcf()
            out = FIG_DIR / f"umap_{color_col}.png"
            fig.savefig(out, dpi=180, bbox_inches="tight")
            plt.close(fig)
            print(f"[SAVED] {out}")


In [ ]:

# =========================
# Dotplot: curated marker panels after annotation
# Uses a small gene subset and log-normalizes only the plotting copy.
# =========================

plot_marker_panels = {
    "Tumor_OPC_NPC": ["PTPRZ1", "BCAN", "OLIG1", "OLIG2", "PDGFRA", "PTN", "NES", "SOX4", "STMN2", "DCX"],
    "Tumor_AC_MES_Hypoxia": ["AQP4", "GFAP", "CLU", "SPARCL1", "CD44", "TGFBI", "CHI3L1", "CA9", "VEGFA", "HILPDA"],
    "Myeloid": ["LYZ", "TYROBP", "C1QA", "C1QB", "C1QC", "TREM2", "CSF1R", "CD74", "HLA-DRA", "FCN1"],
    "Lymphoid": ["CD3D", "CD3E", "TRAC", "LCK", "NKG7", "GNLY", "GZMB", "CD79A", "MZB1", "IGHM"],
    "Vascular_Stromal": ["PECAM1", "VWF", "KDR", "ESAM", "PDGFRB", "ACTA2", "RGS5", "COL1A1", "COL3A1", "LUM"],
    "Other_rare": ["MOG", "MAG", "MOBP", "HBA1", "HBA2", "HBB", "KIT", "GATA2", "RGS13"],
}

flat_genes = []
for genes in plot_marker_panels.values():
    flat_genes.extend(genes)
flat_genes = list(dict.fromkeys(flat_genes))
present = present_genes(flat_genes, adata.var_names)
missing = [g for g in flat_genes if g not in present]
print("plot marker genes present:", len(present))
print("plot marker genes missing:", missing)

plot_panels_present = {k: present_genes(v, adata.var_names) for k, v in plot_marker_panels.items()}
plot_panels_present = {k: v for k, v in plot_panels_present.items() if len(v) > 0}

if len(present) > 0:
    plot_adata = adata[:, present].copy()
    if "counts" in plot_adata.layers:
        plot_adata.X = plot_adata.layers["counts"].copy()
    sc.pp.normalize_total(plot_adata, target_sum=1e4)
    sc.pp.log1p(plot_adata)

    sc.pl.dotplot(
        plot_adata,
        var_names=plot_panels_present,
        groupby=ANNOTATION_COL,
        standard_scale="var",
        dendrogram=False,
        show=False,
    )
    fig = plt.gcf()
    out = FIG_DIR / "dotplot_marker_panels_by_annotation.png"
    fig.savefig(out, dpi=180, bbox_inches="tight")
    plt.close(fig)
    print(f"[SAVED] {out}")

    sc.pl.dotplot(
        plot_adata,
        var_names=plot_panels_present,
        groupby=CLUSTER_KEY,
        standard_scale="var",
        dendrogram=False,
        show=False,
    )
    fig = plt.gcf()
    out = FIG_DIR / "dotplot_marker_panels_by_cluster.png"
    fig.savefig(out, dpi=180, bbox_inches="tight")
    plt.close(fig)
    print(f"[SAVED] {out}")
else:
    print("[WARNING] no plot marker genes present; skipping dotplots")


In [ ]:

# =========================
# Save annotated h5ad and output manifest
# =========================

# Avoid h5ad write failures due to old rank_genes_groups objects.
clean_old_rank_keys(adata)

adata.uns["marker_based_annotation_HVG8000_r0_6"] = {
    "cluster_key": CLUSTER_KEY,
    "source_cluster_key": SOURCE_CLUSTER_KEY,
    "annotation_col": ANNOTATION_COL,
    "lineage_col": LINEAGE_COL,
    "confidence_col": CONFIDENCE_COL,
    "malignancy_col": MALIGNANCY_COL,
    "caution_col": CAUTION_COL,
    "note_col": NOTE_COL,
    "principle": "Tumor-like labels are putative until CNV validation.",
}

write_h5ad_replace(adata, ANNOTATED_H5AD)
write_h5ad_replace(adata, ANNOTATED_H5AD_ALIAS)

manifest_items = [
    ("annotated_h5ad", ANNOTATED_H5AD),
    ("annotated_h5ad_alias", ANNOTATED_H5AD_ALIAS),
    ("annotation_tsv", ANNOTATION_TSV),
    ("annotation_evidence_tsv", ANNOTATION_EVIDENCE_TSV),
    ("annotation_counts_tsv", ANNOTATION_COUNTS_TSV),
    ("annotation_composition_tsv", ANNOTATION_COMPOSITION_TSV),
    ("annotation_bias_tsv", ANNOTATION_BIAS_TSV),
    ("annotation_decision_report_tsv", ANNOTATION_DECISION_REPORT_TSV),
    ("annotation_marker_panel_presence_tsv", PANEL_PRESENCE_TSV),
    ("figures_dir", FIG_DIR),
]

manifest = pd.DataFrame([
    {"item": name, "path": str(path), "exists": Path(path).exists() if name != "figures_dir" else Path(path).is_dir()}
    for name, path in manifest_items
])
write_tsv_replace(manifest, ANNOTATION_OUTPUT_MANIFEST_TSV)
display(manifest)

print("Done.")
